# Telco Customer Churn Prediction with Decision Tree 🌳

This project applies a **Decision Tree Classifier** to predict whether a telecom customer will churn.

The workflow covers:
- Data Collection
- Data Exploration
- Data Preparation
- Encoding
- Train/Test Split
- Decision Tree Modelling
- Overfitting Detection
- Cost-Complexity Post-Pruning
- Model Evaluation

**Dataset:** IBM Telco Customer Churn from Kaggle  
**Problem Type:** Supervised Learning — Binary Classification

## 1. Import Libraries

In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

## 2. Download and Load the Dataset

The dataset is downloaded directly from Kaggle using `kagglehub`.

In [16]:
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Dataset path:", path)
print("Files:", os.listdir(path))

csv_file = os.path.join(path, os.listdir(path)[0])
df = pd.read_csv(csv_file)

df.head()

Dataset path: C:\Users\LENOVO\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1
Files: ['WA_Fn-UseC_-Telco-Customer-Churn.csv']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Data Exploration

The goal of this stage is to understand the structure, data types, missing values, and class distribution before modifying the data.

In [17]:
print("Shape:", df.shape)
df.info()

Shape: (7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   s

In [18]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [19]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [21]:
df["SeniorCitizen"].value_counts()

SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64

### Important Exploration Finding

`TotalCharges` appears as a text column even though it should be numeric.  
A direct `isnull()` check does not reveal the issue, so invalid numeric values are detected explicitly.

In [22]:
print("TotalCharges dtype:", df["TotalCharges"].dtype)

invalid_total_charges = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
).isna().sum()

print("Invalid TotalCharges values:", invalid_total_charges)

TotalCharges dtype: str
Invalid TotalCharges values: 11


## 4. Data Preparation

### Decisions made
- Remove `customerID` because it is an identifier and does not provide useful predictive information.
- Convert `TotalCharges` to numeric.
- Remove the 11 rows where `TotalCharges` cannot be converted to a valid number.
- Encode binary categorical variables as `0/1`.
- Apply One-Hot Encoding to categorical variables with more than two categories.

In [23]:
df = df.drop(columns=["customerID"])

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df = df.dropna(subset=["TotalCharges"])

print("Shape after cleaning:", df.shape)
print("Missing values:", df.isnull().sum().sum())

Shape after cleaning: (7032, 20)
Missing values: 0


## 5. Binary Encoding

In [24]:
df["gender"] = df["gender"].map({"Male": 0, "Female": 1})
df["PhoneService"] = df["PhoneService"].map({"Yes": 1, "No": 0})
df["Dependents"] = df["Dependents"].map({"Yes": 1, "No": 0})
df["Partner"] = df["Partner"].map({"Yes": 1, "No": 0})
df["PaperlessBilling"] = df["PaperlessBilling"].map({"Yes": 1, "No": 0})
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

## 6. One-Hot Encoding

In [25]:
one_hot_columns = [
    "Contract",
    "OnlineSecurity",
    "OnlineBackup",
    "InternetService",
    "MultipleLines",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "PaymentMethod"
]

df = pd.get_dummies(
    df,
    columns=one_hot_columns,
    dtype=int
)

### Final Data Validation

In [26]:
print("Final shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())
print("Remaining object/string columns:")
print(df.select_dtypes(include=["object", "string"]).columns.tolist())

df.head()

Final shape: (7032, 41)
Remaining missing values: 0
Remaining object/string columns:
[]


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,0,1,0,1,0,1,29.85,29.85,0,...,1,0,0,1,0,0,0,0,1,0
1,0,0,0,0,34,1,0,56.95,1889.50,0,...,1,0,0,1,0,0,0,0,0,1
2,0,0,0,0,2,1,1,53.85,108.15,1,...,1,0,0,1,0,0,0,0,0,1
3,0,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,1,0,0,1,0,0,0
4,1,0,0,0,2,1,1,70.70,151.65,1,...,1,0,0,1,0,0,0,0,1,0


## 7. Split Features and Target

`Churn` is the target variable (`y`).  
All remaining columns are used as input features (`X`).

In [27]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

## 8. Train/Test Split

An 80/20 split is used.  
`stratify=y` keeps the churn class proportions similar in both the training and testing sets.

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5625, 40)
X_test : (1407, 40)
y_train: (5625,)
y_test : (1407,)


## 9. Baseline Decision Tree

First, an unrestricted Decision Tree is trained to observe its default behavior.

In [29]:
baseline_model = DecisionTreeClassifier(
    random_state=42
)

baseline_model.fit(X_train, y_train)

baseline_train_accuracy = baseline_model.score(X_train, y_train)
baseline_test_accuracy = baseline_model.score(X_test, y_test)

print("Baseline Train Accuracy:", baseline_train_accuracy)
print("Baseline Test Accuracy :", baseline_test_accuracy)

Baseline Train Accuracy: 0.9987555555555555
Baseline Test Accuracy : 0.728500355366027


### Baseline Result

Observed result:

- **Train Accuracy:** ~99.88%
- **Test Accuracy:** ~72.00%

The large gap is a strong sign of **overfitting**.  
The tree learned the training data extremely well but generalized poorly to unseen customers.

## 10. Post-Pruning with Cost-Complexity Pruning

To reduce overfitting, the Decision Tree is pruned using `ccp_alpha`.

The pruning path provides candidate alpha values.

In [30]:
pruning_path = baseline_model.cost_complexity_pruning_path(
    X_train,
    y_train
)

ccp_alphas = pruning_path.ccp_alphas

print("Number of candidate alphas:", len(ccp_alphas))

Number of candidate alphas: 446


A validation split is created from the training data so that the final test set remains untouched during model selection.

In [31]:
X_train2, X_val, y_train2, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

sample_alphas = np.linspace(
    ccp_alphas.min(),
    ccp_alphas.max(),
    30
)

train_scores = []
val_scores = []

for alpha in sample_alphas:
    clf = DecisionTreeClassifier(
        random_state=42,
        ccp_alpha=alpha
    )

    clf.fit(X_train2, y_train2)

    train_scores.append(clf.score(X_train2, y_train2))
    val_scores.append(clf.score(X_val, y_val))

best_index = np.argmax(val_scores)
best_alpha = sample_alphas[best_index]

print("Best alpha:", best_alpha)
print("Train Accuracy:", train_scores[best_index])
print("Validation Accuracy:", val_scores[best_index])

Best alpha: 0.004561814236319019
Train Accuracy: 0.792
Validation Accuracy: 0.784


### Selected Pruning Strength

Observed during the project:

- **Best `ccp_alpha`:** ~0.00456
- **Training Accuracy:** ~79.2%
- **Validation Accuracy:** ~78.4%

The training and validation scores became much closer, indicating a major reduction in overfitting.

## 11. Train the Final Pruned Model

In [32]:
final_model = DecisionTreeClassifier(
    random_state=42,
    ccp_alpha=best_alpha
)

final_model.fit(X_train, y_train)

final_train_accuracy = final_model.score(X_train, y_train)
final_test_accuracy = final_model.score(X_test, y_test)

print("Final Train Accuracy:", final_train_accuracy)
print("Final Test Accuracy :", final_test_accuracy)

Final Train Accuracy: 0.7907555555555555
Final Test Accuracy : 0.7782515991471215


### Final Accuracy Comparison

| Model | Train Accuracy | Test Accuracy |
|---|---:|---:|
| Unpruned Decision Tree | ~99.88% | ~72.00% |
| Post-Pruned Decision Tree | ~79.08% | ~77.83% |

Post-pruning reduced the train/test gap substantially and improved generalization.

## 12. Model Evaluation

Accuracy alone is not enough because the churn classes are not perfectly balanced.

We therefore evaluate the model using:
- Confusion Matrix
- Precision
- Recall
- F1-score

In [33]:
y_pred = final_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[964  69]
 [243 131]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.93      0.86      1033
           1       0.66      0.35      0.46       374

    accuracy                           0.78      1407
   macro avg       0.73      0.64      0.66      1407
weighted avg       0.76      0.78      0.75      1407



### Observed Confusion Matrix

```text
[[964,  69],
 [243, 131]]
```

Interpretation:

- **TN = 964:** correctly predicted non-churn customers.
- **FP = 69:** predicted churn, but the customer did not churn.
- **FN = 243:** customer actually churned, but the model predicted no churn.
- **TP = 131:** correctly predicted churn customers.

The model performs much better on the non-churn class than on the churn class.

## 13. Key Evaluation Results

Observed classification performance:

| Metric | Class 0 | Class 1 |
|---|---:|---:|
| Precision | 0.80 | 0.66 |
| Recall | 0.93 | 0.35 |
| F1-score | 0.86 | 0.46 |

**Overall Accuracy:** ~78%

The most important weakness is the low **Recall for Churn = 1**.  
The model detects only about 35% of customers who actually churn.

This demonstrates why **accuracy alone can be misleading** in classification problems.

## 14. Conclusion

This project demonstrated a complete supervised machine-learning workflow using a Decision Tree classifier.

### What was learned

- How to explore a real-world dataset before modelling.
- Why data types must be checked instead of relying only on `isnull()`.
- How to prepare categorical data using binary and One-Hot Encoding.
- How Decision Trees can severely overfit when left unrestricted.
- How Cost-Complexity Post-Pruning can reduce overfitting.
- Why classification evaluation requires more than accuracy.
- How Precision, Recall, F1-score, and the Confusion Matrix reveal model weaknesses.

### Final Takeaway

Post-pruning improved the model's generalization substantially, increasing test accuracy from approximately **72% to 78%** while reducing the large train/test performance gap.

However, the model still has relatively low recall for churn customers, showing that there is room for future improvement using techniques beyond the current project scope.

## 15. Future Improvements

Possible future work after learning additional techniques:

- Hyperparameter tuning
- Class-weight adjustment
- Resampling techniques
- Threshold tuning
- Random Forest / ensemble methods
- Cross-validation

These were intentionally left outside the current project so the notebook remains focused on the Decision Tree concepts covered in the course.